# RNN_Based NMT

### Preparation

In [ ]:
import json
import math
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

try:
    import jieba
except ImportError:
    jieba = None

try:
    import sacrebleu
except ImportError:
    sacrebleu = None


seed = 250010155
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

### Tokenization & Vocab

In [20]:
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = SPECIALS

def tokenize_zh(s: str) -> List[str]:
    s = s.strip()
    if jieba is None:
        # fallback: char-level
        return list(s)
    return list(jieba.cut(s, cut_all=False))

def tokenize_en(s: str) -> List[str]:
    # simple whitespace tokenization
    return s.strip().split()

class Vocab:
    def __init__(self, min_freq: int = 2, max_size: int = 50000):
        self.min_freq = min_freq
        self.max_size = max_size
        self.freq: Dict[str, int] = {}
        self.stoi: Dict[str, int] = {}
        self.itos: List[str] = []

    def add_sentence(self, tokens: List[str]):
        for t in tokens:
            self.freq[t] = self.freq.get(t, 0) + 1

    def build(self):
        items = [(t, c) for t, c in self.freq.items() if c >= self.min_freq]
        items.sort(key=lambda x: (-x[1], x[0]))
        items = items[: max(0, self.max_size - len(SPECIALS))]

        self.itos = SPECIALS + [t for t, _ in items]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def encode(self, tokens: List[str]) -> List[int]:
        return [self.stoi.get(t, self.stoi[UNK]) for t in tokens]

    def decode(self, ids: List[int], stop_at_eos: bool = True) -> List[str]:
        out = []
        for i in ids:
            if i < 0 or i >= len(self.itos):
                tok = UNK
            else:
                tok = self.itos[i]
            if stop_at_eos and tok == EOS:
                break
            out.append(tok)
        return out

    @property
    def pad_id(self) -> int: return self.stoi[PAD]
    @property
    def bos_id(self) -> int: return self.stoi[BOS]
    @property
    def eos_id(self) -> int: return self.stoi[EOS]
    @property
    def unk_id(self) -> int: return self.stoi[UNK]
    def __len__(self): return len(self.itos)

### Dataset

In [19]:
class JsonlParallelDataset(Dataset):
    def __init__(
        self,
        path: str,
        src_key: str,
        tgt_key: str,
        src_tok,
        tgt_tok,
        src_vocab: Optional[Vocab] = None,
        tgt_vocab: Optional[Vocab] = None,
        max_len: int = 120,
        build_vocab: bool = False,
    ):
        self.data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                src = obj[src_key]
                tgt = obj[tgt_key]
                src_tokens = src_tok(src)
                tgt_tokens = tgt_tok(tgt)
                if len(src_tokens) == 0 or len(tgt_tokens) == 0:
                    continue
                if len(src_tokens) > max_len or len(tgt_tokens) > max_len:
                    continue
                self.data.append((src_tokens, tgt_tokens))

        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

        if build_vocab:
            assert src_vocab is not None and tgt_vocab is not None
            for s, t in self.data:
                src_vocab.add_sentence(s)
                tgt_vocab.add_sentence(t)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch, src_vocab: Vocab, tgt_vocab: Vocab, device: torch.device):
    src_ids = []
    tgt_ids = []
    for src_tok, tgt_tok in batch:
        s = [src_vocab.bos_id] + src_vocab.encode(src_tok) + [src_vocab.eos_id]
        t = [tgt_vocab.bos_id] + tgt_vocab.encode(tgt_tok) + [tgt_vocab.eos_id]
        src_ids.append(torch.tensor(s, dtype=torch.long))
        tgt_ids.append(torch.tensor(t, dtype=torch.long))

    src_lens = torch.tensor([len(x) for x in src_ids], dtype=torch.long)
    tgt_lens = torch.tensor([len(x) for x in tgt_ids], dtype=torch.long)

    src_pad = pad_sequence(src_ids, batch_first=True, padding_value=src_vocab.pad_id).to(device)
    tgt_pad = pad_sequence(tgt_ids, batch_first=True, padding_value=tgt_vocab.pad_id).to(device)

    return src_pad, src_lens.to(device), tgt_pad, tgt_lens.to(device)


### Attention

In [18]:
class Attention(nn.Module):
    """
    alignment:
      - dot: score = h_t^T s_i (requires same dim)
      - general: score = h_t^T W s_i (multiplicative)
      - additive: score = v^T tanh(W_h h_t + W_s s_i) (Bahdanau)
    """
    def __init__(self, alignment: str, hidden_size: int):
        super().__init__()
        assert alignment in ["dot", "general", "additive"]
        self.alignment = alignment
        self.hidden_size = hidden_size

        if alignment == "general":
            self.W = nn.Linear(hidden_size, hidden_size, bias=False)
        elif alignment == "additive":
            self.W_h = nn.Linear(hidden_size, hidden_size, bias=False)
            self.W_s = nn.Linear(hidden_size, hidden_size, bias=False)
            self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, query: torch.Tensor, keys: torch.Tensor, mask: torch.Tensor):
        """
        query: [B, H]   (decoder state at time t)
        keys:  [B, S, H] (encoder outputs)
        mask:  [B, S]  True for valid, False for pad
        returns:
          context: [B, H]
          attn:    [B, S]
        """
        B, S, H = keys.shape

        if self.alignment == "dot":
            scores = torch.bmm(keys, query.unsqueeze(2)).squeeze(2)  # [B, S]
        elif self.alignment == "general":
            proj = self.W(keys)  # [B, S, H]
            scores = torch.bmm(proj, query.unsqueeze(2)).squeeze(2)
        else:
            # additive
            # keys: [B,S,H], query: [B,H] -> [B,1,H] broadcast
            e = torch.tanh(self.W_s(keys) + self.W_h(query).unsqueeze(1))  # [B,S,H]
            scores = self.v(e).squeeze(2)  # [B,S]

        scores = scores.masked_fill(~mask, float("-inf"))
        attn = F.softmax(scores, dim=1)  # [B,S]
        context = torch.bmm(attn.unsqueeze(1), keys).squeeze(1)  # [B,H]
        return context, attn

### Encoder & Decoder

In [32]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int, hidden_size: int, num_layers: int, rnn_type: str, dropout: float):
        super().__init__()
        assert rnn_type in ["gru", "lstm"]
        self.rnn_type = rnn_type
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        rnn_cls = nn.GRU if rnn_type == "gru" else nn.LSTM
        self.rnn = rnn_cls(
            input_size=emb_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, src_ids: torch.Tensor, src_lens: torch.Tensor):
        # src_ids: [B,S]
        emb = self.embedding(src_ids)  # [B,S,E]
        packed = pack_padded_sequence(emb, src_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, h = self.rnn(packed)
        out, _ = pad_packed_sequence(
                    packed_out,
                    batch_first=True,
                    total_length=src_ids.size(1)
                )
        return out, h  # h: GRU [L,B,H], LSTM tuple


class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_size: int,
        hidden_size: int,
        num_layers: int,
        rnn_type: str,
        dropout: float,
        attn: Attention,
    ):
        super().__init__()
        assert rnn_type in ["gru", "lstm"]
        self.rnn_type = rnn_type
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        self.attn = attn

        rnn_cls = nn.GRU if rnn_type == "gru" else nn.LSTM
        # input = [emb ; context]
        self.rnn = rnn_cls(
            input_size=emb_size + hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.out = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(
        self,
        y_prev: torch.Tensor,      # [B] token id at time t-1
        state,                      # GRU: [L,B,H] or LSTM: (h,c)
        enc_out: torch.Tensor,      # [B,S,H]
        enc_mask: torch.Tensor,     # [B,S]
    ):
        emb = self.dropout(self.embedding(y_prev)).unsqueeze(1)  # [B,1,E]

        # query = top layer hidden at t-1
        if self.rnn_type == "gru":
            query = state[-1]  # [B,H]
        else:
            query = state[0][-1]  # h[-1]

        ctx, attn = self.attn(query, enc_out, enc_mask)  # ctx [B,H]
        ctx = ctx.unsqueeze(1)  # [B,1,H]

        rnn_in = torch.cat([emb, ctx], dim=2)  # [B,1,E+H]
        out, new_state = self.rnn(rnn_in, state)  # out [B,1,H]
        logits = self.out(out.squeeze(1))  # [B,V]
        return logits, new_state, attn


class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_pad_id: int):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_id = src_pad_id

    def forward(
        self,
        src_ids: torch.Tensor,
        src_lens: torch.Tensor,
        tgt_ids: torch.Tensor,
        tf_ratio: float = 1.0,
    ):
        """
        Train-time forward.
        tgt_ids: [B,T] includes BOS ... EOS
        returns logits: [B, T-1, V] predicting tgt at 1..T-1
        """
        enc_out, enc_state = self.encoder(src_ids, src_lens)
        B, S, H = enc_out.shape
        enc_mask = (src_ids != self.src_pad_id)  # [B,S]

        # init decoder state as encoder final state (same shape)
        dec_state = enc_state

        T = tgt_ids.size(1)
        logits_all = []
        y_prev = tgt_ids[:, 0]  # BOS

        for t in range(1, T):
            logits, dec_state, _ = self.decoder.forward_step(y_prev, dec_state, enc_out, enc_mask)
            logits_all.append(logits.unsqueeze(1))

            use_tf = (random.random() < tf_ratio)
            if use_tf:
                y_prev = tgt_ids[:, t]
            else:
                y_prev = torch.argmax(logits, dim=1)

        return torch.cat(logits_all, dim=1)  # [B,T-1,V]

    @torch.no_grad()
    def translate_greedy(self, src_ids: torch.Tensor, src_lens: torch.Tensor, bos_id: int, eos_id: int, max_len: int = 120):
        enc_out, enc_state = self.encoder(src_ids, src_lens)
        enc_mask = (src_ids != self.src_pad_id)
        dec_state = enc_state

        B = src_ids.size(0)
        y_prev = torch.full((B,), bos_id, dtype=torch.long, device=src_ids.device)
        outputs = [[] for _ in range(B)]

        for step in range(max_len):
            logits, dec_state, _ = self.decoder.forward_step(y_prev, dec_state, enc_out, enc_mask)
            if step == 0:
                logits[:, eos_id] = -1e9
            y_prev = torch.argmax(logits, dim=1)
            for i in range(B):
                outputs[i].append(int(y_prev[i].item()))
            if all((len(o) > 0 and o[-1] == eos_id) for o in outputs):
                break
        return outputs

    @torch.no_grad()
    def translate_beam(
        self,
        src_ids: torch.Tensor,
        src_lens: torch.Tensor,
        bos_id: int,
        eos_id: int,
        beam_size: int = 5,
        max_len: int = 120,
        len_norm_alpha: float = 0.6,
    ):
        """
        Beam search (batch size = 1 for simplicity & robustness).
        If you want batch beam, we can do it next.
        """
        assert src_ids.size(0) == 1, "Beam search here assumes batch_size=1."
        device = src_ids.device

        enc_out, enc_state = self.encoder(src_ids, src_lens)
        enc_mask = (src_ids != self.src_pad_id)

        # Each beam: (tokens, state, logprob, ended)
        beams = [([bos_id], enc_state, 0.0, False)]

        for _ in range(max_len):
            new_beams = []
            all_ended = True

            for tokens, state, lp, ended in beams:
                if ended:
                    new_beams.append((tokens, state, lp, True))
                    continue

                all_ended = False
                y_prev = torch.tensor([tokens[-1]], dtype=torch.long, device=device)
                logits, new_state, _ = self.decoder.forward_step(y_prev, state, enc_out, enc_mask)
                logp = F.log_softmax(logits, dim=1).squeeze(0)  # [V]

                if len(tokens) == 1:
                    logp[eos_id] = float("-inf")

                topk = torch.topk(logp, k=beam_size)
                for k in range(beam_size):
                    tok = int(topk.indices[k].item())
                    score = float(topk.values[k].item())
                    ntoks = tokens + [tok]
                    nlp = lp + score
                    nended = (tok == eos_id)
                    new_beams.append((ntoks, new_state, nlp, nended))

            if all_ended:
                break

            # length normalization
            def norm_score(b):
                tokens, _, lp, _ = b
                L = max(1, len(tokens) - 1)  # exclude BOS
                return lp / ((5 + L) ** len_norm_alpha / (5 ** len_norm_alpha))

            new_beams.sort(key=norm_score, reverse=True)
            beams = new_beams[:beam_size]

        best = max(beams, key=lambda b: b[2])
        # remove BOS
        return [best[0][1:]]


### Training & Eval

In [34]:
@dataclass
class TrainConfig:
    lr: float = 3e-4
    clip: float = 1.0
    label_smooth: float = 0.0

def loss_fn(logits: torch.Tensor, tgt: torch.Tensor, pad_id: int, label_smooth: float = 0.0):
    """
    logits: [B, T, V], tgt: [B, T]
    """
    B, T, V = logits.shape
    logits = logits.reshape(B * T, V)
    tgt = tgt.reshape(B * T)

    if label_smooth <= 0:
        return F.cross_entropy(logits, tgt, ignore_index=pad_id)

    # simple label smoothing
    logp = F.log_softmax(logits, dim=1)
    nll = F.nll_loss(logp, tgt, ignore_index=pad_id, reduction="none")
    smooth = -logp.mean(dim=1)
    mask = (tgt != pad_id).float()
    nll = (nll * mask).sum() / mask.sum().clamp_min(1.0)
    smooth = (smooth * mask).sum() / mask.sum().clamp_min(1.0)
    return (1 - label_smooth) * nll + label_smooth * smooth

@torch.no_grad()
def eval_bleu(model: Seq2Seq, loader: DataLoader, tgt_vocab: Vocab, bos_id: int, eos_id: int, decode: str, beam_size: int):
    if sacrebleu is None:
        print("[WARN] sacrebleu not installed; BLEU skipped.")
        return None

    FILTER = {"<pad>", "<bos>", "<eos>"}

    model.eval()
    hyps = []
    refs = []
    for src_ids, src_lens, tgt_ids, _ in tqdm(loader, desc="eval", leave=False):
        # refs: decode target removing BOS
        for b in range(tgt_ids.size(0)):
            ref_tokens = tgt_vocab.decode(tgt_ids[b].tolist()[1:], stop_at_eos=True)
            refs.append(" ".join([t for t in ref_tokens if t not in FILTER]))

        if decode == "greedy":
            out_ids = model.translate_greedy(src_ids, src_lens, bos_id, eos_id)
        else:
            # beam search defined for batch=1; so do per-sample
            out_ids = []
            for b in range(src_ids.size(0)):
                hyp = model.translate_beam(
                    src_ids[b:b+1], src_lens[b:b+1], bos_id, eos_id, beam_size=beam_size
                )[0]
                out_ids.append(hyp)

        for b in range(len(out_ids)):
            hyp_tokens = tgt_vocab.decode(out_ids[b], stop_at_eos=True)
            hyp = " ".join([t for t in hyp_tokens if t not in FILTER])
            hyps.append(hyp)

    if len(hyps) <= 3:   # 只看前 3 条
        src_tokens = tgt_vocab.decode(tgt_ids[b].tolist()[1:], stop_at_eos=True)

        print("SRC:", " ".join(src_tokens[:80]))
        print("REF:", refs[-1][:200])
        print("HYP:", hyp[:200])
        print("-" * 50)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])

    empty_h = sum(1 for h in hyps if len(h.strip()) == 0)
    empty_r = sum(1 for r in refs if len(r.strip()) == 0)

    print("BLEU:", bleu.score)
    #print("signature:", bleu.signature)
    print("empty_hyp:", empty_h, "empty_ref:", empty_r, "total:", len(hyps))

    return bleu.score

def train_one_epoch(model: Seq2Seq, loader: DataLoader, optim, pad_id: int, tf_ratio: float, cfg: TrainConfig):
    model.train()
    total_loss = 0.0
    n_steps = 0

    for src_ids, src_lens, tgt_ids, _ in tqdm(loader, desc="train", leave=False):
        optim.zero_grad()
        logits = model(src_ids, src_lens, tgt_ids, tf_ratio=tf_ratio)  # [B,T-1,V]
        tgt_gold = tgt_ids[:, 1:]  # predict next tokens
        loss = loss_fn(logits, tgt_gold, pad_id=pad_id, label_smooth=cfg.label_smooth)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.clip)
        optim.step()

        total_loss += float(loss.item())
        n_steps += 1

    return total_loss / max(1, n_steps)

### Main

In [35]:
def build_model(args, src_vocab: Vocab, tgt_vocab: Vocab) -> Seq2Seq:
    enc = Encoder(
        vocab_size=len(src_vocab),
        emb_size=args.emb,
        hidden_size=args.hid,
        num_layers=2,
        rnn_type=args.rnn,
        dropout=args.dropout,
    )
    attn = Attention(args.attn, hidden_size=args.hid)
    dec = Decoder(
        vocab_size=len(tgt_vocab),
        emb_size=args.emb,
        hidden_size=args.hid,
        num_layers=2,
        rnn_type=args.rnn,
        dropout=args.dropout,
        attn=attn,
    )
    return Seq2Seq(enc, dec, src_pad_id=src_vocab.pad_id)


def run_train(args):
    device = torch.device(args.device)

    # 1) Build vocab
    src_vocab = Vocab(min_freq=args.min_freq, max_size=args.vocab_size)
    tgt_vocab = Vocab(min_freq=args.min_freq, max_size=args.vocab_size)

    train_ds_for_vocab = JsonlParallelDataset(
        args.train, args.src_key, args.tgt_key,
        src_tok=tokenize_zh, tgt_tok=tokenize_en,
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        max_len=args.max_len, build_vocab=True
    )
    src_vocab.build()
    tgt_vocab.build()

    print(f"src_vocab={len(src_vocab)}, tgt_vocab={len(tgt_vocab)}, train_samples={len(train_ds_for_vocab)}")

    # 2) Datasets
    train_ds = train_ds_for_vocab
    valid_ds = JsonlParallelDataset(
        args.valid, args.src_key, args.tgt_key,
        src_tok=tokenize_zh, tgt_tok=tokenize_en,
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        max_len=args.max_len, build_vocab=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=args.batch, shuffle=True,
        collate_fn=lambda b: collate_fn(b, src_vocab, tgt_vocab, device),
        num_workers=0
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=min(args.batch, 64), shuffle=False,
        collate_fn=lambda b: collate_fn(b, src_vocab, tgt_vocab, device),
        num_workers=0
    )

    # 3) Model
    model = build_model(args, src_vocab, tgt_vocab).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=args.lr)
    cfg = TrainConfig(lr=args.lr, clip=args.clip, label_smooth=args.label_smooth)

    best_bleu = -1.0
    for ep in range(1, args.epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optim, pad_id=tgt_vocab.pad_id, tf_ratio=args.tf_ratio, cfg=cfg)
        bleu = eval_bleu(model, valid_loader, tgt_vocab, tgt_vocab.bos_id, tgt_vocab.eos_id, args.decode, args.beam_size)
        bleu_str = "N/A" if bleu is None else f"{bleu:.2f}"

        print(f"[epoch {ep}] train_loss={tr_loss:.4f} valid_BLEU={bleu_str}")

        score = -tr_loss if bleu is None else bleu
        if score > best_bleu:
            best_bleu = score
            ckpt = {
                "args": vars(args),
                "src_vocab": {"itos": src_vocab.itos, "stoi": src_vocab.stoi},
                "tgt_vocab": {"itos": tgt_vocab.itos, "stoi": tgt_vocab.stoi},
                "model": model.state_dict(),
            }
            torch.save(ckpt, args.save)
            print(f"  saved: {args.save}")
    return model, (src_vocab, tgt_vocab)

In [54]:
from types import SimpleNamespace
import torch

args = SimpleNamespace(
    train="data/train_10k.jsonl",
    valid="data/valid.jsonl",
    src_key="zh",
    tgt_key="en",
    max_len=120,
    min_freq=2,
    vocab_size=50000,

    rnn="gru",                   # "lstm"
    attn="additive",             # "dot" / "general" / "additive"
    emb=256,
    hid=256,
    dropout=0.2,

    batch=64,
    epochs=10,
    lr=1e-3,
    clip=1.0,
    label_smooth=0.0,

    tf_ratio=1.0,
    decode="beam",
    beam_size=5,

    device="cuda" if torch.cuda.is_available() else "cpu",
    save="rnn_nmt.pt",
)


In [55]:
model, (src_vocab, tgt_vocab) = run_train(args)

src_vocab=9687, tgt_vocab=12485, train_samples=9998


BLEU: 16.791044189034444
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 1] train_loss=6.9708 valid_BLEU=16.79
  saved: rnn_nmt.pt


BLEU: 20.83286455876483
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 2] train_loss=6.5188 valid_BLEU=20.83
  saved: rnn_nmt.pt


BLEU: 21.91732225495134
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 3] train_loss=6.2027 valid_BLEU=21.92
  saved: rnn_nmt.pt


BLEU: 19.07377345243531
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 4] train_loss=5.9011 valid_BLEU=19.07


...

BLEU: 20.2637749813462
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 6] train_loss=5.3968 valid_BLEU=20.26


BLEU: 19.11081023002276
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 7] train_loss=5.1770 valid_BLEU=19.11


BLEU: 18.458730208099652
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 8] train_loss=4.9723 valid_BLEU=18.46


BLEU: 19.21660597706637
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 9] train_loss=4.7804 valid_BLEU=19.22


BLEU: 17.024294635353506
empty_hyp: 0 empty_ref: 0 total: 500
[epoch 10] train_loss=4.5995 valid_BLEU=17.02
